In [1]:
import pandas as pd
import os 

In [2]:
usuario = os.getlogin() 

In [7]:
base = pd.read_excel(fr"C:\Users\{usuario}\Downloads\nuevo-f\formulario-ang\bases\UnidadesIMB_CS!_v2.xlsx",sheet_name="Sheet 1")
clues = pd.read_parquet(fr"C:\Users\{usuario}\IMSS-BIENESTAR\División de Procesamiento de información - Repositorio de Datos\CLUES\clues.parquet")

In [ ]:
# import pandas as pd

# sheet_id = "1maRNGDuU9rEFWZLgMdhJS1waAnJxl6ENntm-nyD0tq8"
# gid = "1765182479"

# url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={gid}"

# base_an = pd.read_csv(url)



HTTPError: HTTP Error 429: Too Many Requests

In [5]:
import requests
import pandas as pd
from io import StringIO

sheet_id = "1maRNGDuU9rEFWZLgMdhJS1waAnJxl6ENntm-nyD0tq8"
gid = "1765182479"

url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={gid}"

# Descargar el archivo
response = requests.get(url, timeout=30)
response.raise_for_status()

# Guardar el CSV
with open("base_an.csv", "wb") as f:
    f.write(response.content)

print("CSV descargado correctamente: base_an.csv")

# Leer el archivo local
base_an = pd.read_csv("base_an.csv")

print(base_an.head())

CSV descargado correctamente: base_an.csv
        fecha_registro tipo_registro          entidad  \
0  15/07/2026 20:33:13        unidad         CAMPECHE   
1  17/07/2026 11:28:26        unidad          HIDALGO   
2   9/07/2026 15:34:57     respuesta  BAJA CALIFORNIA   
3   8/07/2026 12:52:10     respuesta  BAJA CALIFORNIA   
4   8/07/2026 12:52:20     respuesta  BAJA CALIFORNIA   

                usuario_nombre                           usuario_email  \
0  Jesús Antonio Santini Pérez                drsantiniperez@gmail.com   
1   Armando Hernandez Libreros  armando.hernandez@imssbienestar.gob.mx   
2              minelly montoya                       minelly@gmail.com   
3              minelly montoya                       minelly@gmail.com   
4              minelly montoya                       minelly@gmail.com   

     clues_imb                     nombre_de_la_unidad  \
0  CCIMB000085  CENTRO DE SALUD DR. WILBERTH ESCALANTE   
1  HGIMB000530       PROGRESO EL DE ATOTONILCO DE TULA

In [8]:
col = ["clues_imb"]
base = base[col]

In [9]:
clues.columns

Index(['clues_imb', 'clues_ssa_y_sme', 'categoria_gerencial_uas',
       'categoria_gerencial', 'categoria_gerencial_ampliada',
       'clave_de_la_entidad', 'entidad', 'clave_del_municipio', 'municipio',
       'localidad', 'clave_de_la_localidad', 'nombre_de_la_unidad',
       'nombre_comercial', 'estatus_de_operacion', 'nivel_atencion',
       'clave_de_tipologia', 'nombre_de_tipologia', 'clave_de_subtipologia',
       'nombre_de_subtipologia', 'estrato_unidad', 'cve_ro', 'nombre_region',
       'latitud', 'longitud', 'organ_ro', 'tipo_hbc'],
      dtype='object')

In [10]:
base = base.merge(
    clues[["clues_imb", "entidad",'nombre_de_la_unidad']],
    on="clues_imb",
    how="left"
)

In [11]:
colum =['clues_imb', 'entidad','consultorio','pregunta','nombre_de_la_unidad']
base_an = base_an[colum]    

In [12]:
b = pd.read_excel(fr"C:\Users\{usuario}\Downloads\nuevo-f\formulario-ang\bases\UM_IMB_SUS.xlsx",sheet_name="Hoja2")

In [13]:
b = b.drop(index=[0, 2, 5,3,1])

In [14]:
b = b.drop(columns=['Unnamed: 1'])

In [15]:
b = b.rename(columns={
   'CLUES' : 'preguntas',
})

In [16]:
b.columns

Index(['preguntas'], dtype='object')

In [17]:
base

,clues_imb,entidad,nombre_de_la_unidad
0,BCIMB000051,BAJA CALIFORNIA,COLONIA LOMA LINDA
1,BCIMB000092,BAJA CALIFORNIA,FRACCIONAMIENTO MAR
2,BCIMB000162,BAJA CALIFORNIA,ISLA DE CEDROS
3,BCIMB000186,BAJA CALIFORNIA,LA MISIÓN
4,BCIMB000191,BAJA CALIFORNIA,REAL DEL CASTILLO
...,...,...,...
2822,GRIMB010244,GUERRERO,CENTRO DE SALUD CON SERVICIOS AMPLIADOS DE CHI...
2823,GRIMB010256,GUERRERO,CENTRO DE SALUD CON SERVICIOS AMPLIADOS DE HUI...
2824,GRIMB010565,GUERRERO,CENTRO DE SALUD CON SERVICIOS AMPLIADOS MARQUELIA
2825,SRIMB001071,SONORA,CENTRO DE SALUD RURAL MIGUEL ALEMÁN


In [18]:
base_an.head()

,clues_imb,entidad,consultorio,pregunta,nombre_de_la_unidad
0,CCIMB000085,CAMPECHE,NaN,internet,CENTRO DE SALUD DR. WILBERTH ESCALANTE
1,HGIMB000530,HIDALGO,NaN,internet,PROGRESO EL DE ATOTONILCO DE TULA
2,BCIMB000092,BAJA CALIFORNIA,1.0,equipo_de_computo_consultorio_1,FRACCIONAMIENTO MAR
3,BCIMB000092,BAJA CALIFORNIA,2.0,equipo_de_computo_consultorio_2,FRACCIONAMIENTO MAR
4,BCIMB000092,BAJA CALIFORNIA,1.0,banco_de_altura_consultorio_1,FRACCIONAMIENTO MAR


In [19]:
base_an = base_an[
    ~base_an["pregunta"].str.contains("internet|turno_consultorio|consultorios_habilitados", case=False, na=False)
]

In [20]:
base_conteo =base["clues_imb"].unique()
base_conteo

array(['BCIMB000051', 'BCIMB000092', 'BCIMB000162', ..., 'GRIMB010565',
       'SRIMB001071', 'TCIMB000704'], shape=(2191,), dtype=object)

In [21]:
total_unidades = base["clues_imb"].nunique()

respondieron = base_an["clues_imb"].nunique()

sin_responder = total_unidades - respondieron

avance_general = round(respondieron / total_unidades * 100, 1)

## tablas

In [22]:
# TABLA 1 - AVANCE POR ENTIDAD

# Unidades esperadas
esperadas = (
    base
    .groupby("entidad")
    .size()
    .reset_index(name="esperadas")
)

# Unidades que respondieron
contestadas = (
    base_an[["entidad","clues_imb"]]
    .drop_duplicates()
    .groupby("entidad")
    .size()
    .reset_index(name="respondieron")
)

tabla_entidades = (
    esperadas
    .merge(contestadas,
           on="entidad",
           how="left")
)

tabla_entidades["respondieron"] = (
    tabla_entidades["respondieron"]
    .fillna(0)
    .astype(int)
)

tabla_entidades["porcentaje"] = (
    tabla_entidades["respondieron"]
    / tabla_entidades["esperadas"]
    *100
).round(1)

tabla_entidades = tabla_entidades.sort_values(
    "porcentaje",
    ascending=False
)


In [23]:
tabla_entidades

,entidad,esperadas,respondieron,porcentaje
15,SAN LUIS POTOSI,33,27,81.8
21,ZACATECAS,23,18,78.3
3,CHIAPAS,25,17,68.0
6,GUERRERO,592,394,66.6
8,MEXICO,306,170,55.6
14,QUINTANA ROO,165,89,53.9
16,SINALOA,120,50,41.7
20,VERACRUZ DE IGNACIO DE LA LLAVE,128,48,37.5
11,NAYARIT,95,35,36.8
2,CAMPECHE,45,16,35.6


In [24]:
base_an

,clues_imb,entidad,consultorio,pregunta,nombre_de_la_unidad
2,BCIMB000092,BAJA CALIFORNIA,1.0,equipo_de_computo_consultorio_1,FRACCIONAMIENTO MAR
3,BCIMB000092,BAJA CALIFORNIA,2.0,equipo_de_computo_consultorio_2,FRACCIONAMIENTO MAR
4,BCIMB000092,BAJA CALIFORNIA,1.0,banco_de_altura_consultorio_1,FRACCIONAMIENTO MAR
5,BCIMB000092,BAJA CALIFORNIA,2.0,banco_de_altura_consultorio_2,FRACCIONAMIENTO MAR
12,BCIMB000536,BAJA CALIFORNIA,1.0,equipo_de_computo_consultorio_1,DURANGO
...,...,...,...,...,...
67168,MCIMB000716,MEXICO,3.0,bote_sanitario_con_pedal_consultorio_3,PALMAR CHICO
67169,MCIMB000716,MEXICO,3.0,bote_sanitario_con_pedal_consultorio_3,PALMAR CHICO
67170,MCIMB000716,MEXICO,3.0,mesa_para_colocar_bascula_pesabebes_consultorio_3,PALMAR CHICO
67171,QRIMB001541,QUINTANA ROO,1.0,estadimetro_pediatrico_consultorio_1,CENTRO DE SALUD RURAL MANUEL CRESENCIO REJÓN


In [25]:
base_an = base_an.drop_duplicates()

In [26]:
# TABLA NUEVA - FALTANTES POR ESTADOS

# Universo esperado de unidades (CLUES) por estado
esperadas_estado = (
    base[["entidad", "clues_imb", "nombre_de_la_unidad"]]
    .dropna(subset=["entidad", "clues_imb"])
    .drop_duplicates()
)

# CLUES que si respondieron (al menos una pregunta)
respondidas_estado = (
    base_an[["clues_imb"]]
    .dropna(subset=["clues_imb"])
    .drop_duplicates()
    .merge(
        base[["clues_imb", "entidad"]].drop_duplicates(),
        on="clues_imb",
        how="left"
    )
    .dropna(subset=["entidad"])
    .drop_duplicates()
)

# Anti-join: faltantes = esperadas - respondidas
faltantes_por_estados = (
    esperadas_estado
    .merge(
        respondidas_estado[["entidad", "clues_imb"]],
        on=["entidad", "clues_imb"],
        how="left",
        indicator=True
    )
    .query('_merge == "left_only"')
    .drop(columns=["_merge"])
    .sort_values(["entidad", "clues_imb"])
    .reset_index(drop=True)
)

# Resumen de CLUES faltantes por estado
tabla_faltantes_por_estados = (
    faltantes_por_estados
    .groupby("entidad", as_index=False)
    .agg(clues_faltantes=("clues_imb", "nunique"))
    .sort_values("clues_faltantes", ascending=False)
)

display(tabla_faltantes_por_estados)
display(faltantes_por_estados.head(100))

,entidad,clues_faltantes
6,GUERRERO,198
4,CIUDAD DE MEXICO,192
17,SONORA,123
18,TABASCO,115
19,TAMAULIPAS,81
0,BAJA CALIFORNIA,67
8,MEXICO,51
1,BAJA CALIFORNIA SUR,50
14,QUINTANA ROO,44
5,COLIMA,42


,entidad,clues_imb,nombre_de_la_unidad
0,BAJA CALIFORNIA,BCIMB000051,COLONIA LOMA LINDA
1,BAJA CALIFORNIA,BCIMB000063,COLONIA POPULAR 2
2,BAJA CALIFORNIA,BCIMB000075,COLONIA POPULAR 89
3,BAJA CALIFORNIA,BCIMB000104,COL. JALISCO
4,BAJA CALIFORNIA,BCIMB000116,COL. LOMITAS INDECO
...,...,...,...
95,BAJA CALIFORNIA SUR,BSIMB000375,CENTRO DE SALUD LOS VENADOS. CABO SAN LUCAS
96,BAJA CALIFORNIA SUR,BSIMB000380,CENTRO DE SALUD MIRAFLORES
97,BAJA CALIFORNIA SUR,BSIMB000392,CENTRO DE SALUD LA PLAYA
98,BAJA CALIFORNIA SUR,BSIMB000404,CENTRO DE SALUD SAN JOSÉ VIEJO


In [27]:
# TABLA 2 - COMPLETITUD POR UNIDAD

# Número de preguntas del formulario
n_preguntas = len(b)

# Número de consultorios por unidad
consultorios = (
    base_an
    .groupby(["clues_imb","entidad",'nombre_de_la_unidad'],as_index=False)
    ["consultorio"]
    .max()
)

consultorios.rename(
    columns={"consultorio":"consultorios"},
    inplace=True
)

# Preguntas respondidas
respondidas = (
    base_an
    .groupby(["clues_imb","entidad",'nombre_de_la_unidad'])
    .size()
    .reset_index(name="respondidas")
)

tabla_unidades = consultorios.merge(
    respondidas,
    on=["clues_imb","entidad",'nombre_de_la_unidad']
)

tabla_unidades["esperadas"] = (
    tabla_unidades["consultorios"]
    * n_preguntas
)

tabla_unidades["porcentaje"] = (
    tabla_unidades["respondidas"]
    / tabla_unidades["esperadas"]
    *100
).round(1)

tabla_unidades = (
    tabla_unidades
    .rename(columns={"clues_imb":"clues"})
    .sort_values("porcentaje")
)

In [28]:
tabla_unidades.head()

,clues,entidad,nombre_de_la_unidad,consultorios,respondidas,esperadas,porcentaje
91,GRIMB001086,GUERRERO,R-02 APAXTLA DE CASTREJON,1.0,1,44.0,2.3
470,MCIMB001812,MEXICO,EMILIANO ZAPATA,1.0,1,44.0,2.3
634,MNIMB003894,MICHOACAN DE OCAMPO,CENTRO DE SALUD SAN JUAN DEL FUERTE,1.0,1,44.0,2.3
733,QRIMB000380,QUINTANA ROO,CENTRO DE SALUD URBANO 03 CHETUMAL,1.0,1,44.0,2.3
694,OCIMB000910,OAXACA,LA JOYA SAN MARTÍN MEXICAPAM,1.0,1,44.0,2.3


In [29]:
# LISTAS PARA JINJA2
entidades = tabla_entidades.to_dict(orient="records")
unidades = tabla_unidades.to_dict(orient="records")

In [30]:
tabla_entidades = (
    tabla_unidades
    .groupby("entidad", as_index=False)
    .agg(
        consultorios=("consultorios","sum"),
        respondidas=("respondidas","sum"),
        esperadas=("esperadas","sum"),
        unidades=("clues","count")
    )
)

tabla_entidades["porcentaje"] = (
    tabla_entidades["respondidas"]
    / tabla_entidades["esperadas"]
    *100
).round(1)

tabla_entidades = tabla_entidades.sort_values(
    "porcentaje",
    ascending=False
)

In [31]:
tabla_entidades

,entidad,consultorios,respondidas,esperadas,unidades,porcentaje
8,MORELOS,23.0,1009,1012.0,20,99.7
17,ZACATECAS,38.0,1655,1672.0,18,99.0
9,NAYARIT,37.0,1561,1628.0,35,95.9
15,TAMAULIPAS,96.0,4036,4224.0,69,95.5
4,COLIMA,24.0,986,1056.0,18,93.4
10,OAXACA,57.0,2330,2508.0,26,92.9
16,VERACRUZ DE IGNACIO DE LA LLAVE,85.0,3458,3740.0,48,92.5
11,QUINTANA ROO,105.0,4229,4620.0,89,91.5
5,GUERRERO,466.0,18256,20504.0,394,89.0
6,MEXICO,451.0,17524,19844.0,170,88.3


In [32]:
# Total de unidades por entidad (base completa)
total_por_entidad = (
    base
    .groupby("entidad")["clues_imb"]
    .nunique()
    .reset_index(name="total_unidades")
)

# Unidades que han respondido al menos una pregunta
respondieron_por_entidad = (
    base_an[["entidad", "clues_imb"]]
    .drop_duplicates()
    .groupby("entidad")["clues_imb"]
    .nunique()
    .reset_index(name="unidades_respondieron")
)

tabla_avance = total_por_entidad.merge(
    respondieron_por_entidad,
    on="entidad",
    how="left"
)

tabla_avance["unidades_respondieron"] = (
    tabla_avance["unidades_respondieron"].fillna(0).astype(int)
)

tabla_avance["porcentaje"] = (
    tabla_avance["unidades_respondieron"]
    / tabla_avance["total_unidades"]
    * 100
).round(1)

tabla_avance = tabla_avance.sort_values("porcentaje", ascending=False)

In [33]:
tabla_avance

,entidad,total_unidades,unidades_respondieron,porcentaje
15,SAN LUIS POTOSI,28,27,96.4
21,ZACATECAS,19,18,94.7
8,MEXICO,221,170,76.9
16,SINALOA,66,50,75.8
3,CHIAPAS,24,17,70.8
14,QUINTANA ROO,133,89,66.9
6,GUERRERO,592,394,66.6
20,VERACRUZ DE IGNACIO DE LA LLAVE,74,48,64.9
11,NAYARIT,56,35,62.5
2,CAMPECHE,31,16,51.6


## graficas

In [34]:
import plotly.express as px

# Tabla base para la grafica
tabla = tabla_avance.sort_values("porcentaje", ascending=False).copy()

# Lista de CLUES faltantes por entidad para hover
faltantes_hover = (
    faltantes_por_estados
    .groupby("entidad")["clues_imb"]
    .apply(lambda s: "<br>".join(sorted(s.astype(str).unique())))
    .reset_index(name="clues_faltantes_hover")
)

tabla = tabla.merge(faltantes_hover, on="entidad", how="left")
tabla["clues_faltantes_hover"] = tabla["clues_faltantes_hover"].fillna("Sin CLUES faltantes")

# Semaforo de colores
tabla["color"] = tabla["porcentaje"].apply(
    lambda p:
        "#0D5D2A" if p == 100 else
        "#88A91E" if p >= 80 else
        "#F1D54A" if p >= 50 else
        "#D41111"
)

fig = px.bar(
    tabla,
    x="entidad",
    y="porcentaje",
    text="porcentaje",
    title="Avance por entidad sobre unidades",
    custom_data=["clues_faltantes_hover"]
)

fig.update_traces(
    marker_color=tabla["color"],
    texttemplate="%{text:.1f}%",
    textposition="outside",
    hovertemplate="<b>%{x}</b><br>CLUES faltantes:<br>%{customdata[0]}<extra></extra>"
)

fig.update_yaxes(showticklabels=False)

fig.update_layout(
    xaxis_title="",
    yaxis_title="",
    template="plotly_white",
    width=1200,
    height=700
)

fig.show()

In [35]:
import plotly.graph_objects as go

# Ordenar de menor a mayor para efecto cascada
df_plot = tabla_entidades.sort_values("porcentaje", ascending=True)

# Semáforo de colores
colores = [
    "#0D5D2A" if p >= 95 else
    "#88A91E" if p >= 80 else
    "#F1D54A" if p >= 60 else
    "#D41111"
    for p in df_plot["porcentaje"]
]

fig_cascada = go.Figure()

fig_cascada.add_trace(
    go.Bar(
        x=df_plot["porcentaje"],
        y=df_plot["entidad"],
        orientation="h",
        marker=dict(color=colores),
        text=[f"{p:.1f}%" for p in df_plot["porcentaje"]],
        textposition="outside",
        textfont=dict(size=11),
        hovertemplate="<b>%{y}</b><br>Porcentaje: %{x:.1f}%<extra></extra>",
    )
)

fig_cascada.update_layout(
    title=dict(
        text="Gráfica de llenado de consultorios",
        x=0.5,
        font=dict(size=16)
    ),
    xaxis=dict(
        title="",
        range=[0, 115],
        ticksuffix="%",
        showgrid=True,
        gridcolor="lightgrey"
    ),
    yaxis=dict(
        title="",
        automargin=True
    ),
    plot_bgcolor="white",
    height=500,
    margin=dict(l=20, r=60, t=60, b=40)
)

fig_cascada.show()

In [36]:
from pathlib import Path
import json
from datetime import datetime
from plotly.utils import PlotlyJSONEncoder

base_dir = Path.cwd()
salida_data = base_dir / "data.js"
html_origen = base_dir / "reporte_interactivo.html"
index_destino = base_dir / "index.html"

figures = [
    {"id": "fig_avance", "figure": fig.to_plotly_json()},
]

if "fig_cascada" in globals():
    figures.append({"id": "fig_cascada", "figure": fig_cascada.to_plotly_json()})
else:
    print("Aviso: fig_cascada no existe todavia; se exporta solo fig_avance.")

payload = {
    "title": "Tablero IMSS Bienestar",
    "updated_at": datetime.now().isoformat(timespec="seconds"),
    "tables": {
        "tabla_avance": {
            "columns": list(tabla_avance.columns),
            "rows": tabla_avance.to_dict(orient="records"),
            "excel": "tabla_avance.xlsx",
        },
        "tabla_entidades": {
            "columns": list(tabla_entidades.columns),
            "rows": tabla_entidades.to_dict(orient="records"),
            "excel": "tabla_entidades.xlsx",
        },
        "tabla_unidades": {
            "columns": list(tabla_unidades.columns),
            "rows": tabla_unidades.to_dict(orient="records"),
            "excel": "tabla_unidades.xlsx",
        },
        "faltantes_por_estados": {
            "columns": list(faltantes_por_estados.columns),
            "rows": faltantes_por_estados.to_dict(orient="records"),
            "excel": "faltantes_por_estados.xlsx",
        },
        "tabla_faltantes_por_estados": {
            "columns": list(tabla_faltantes_por_estados.columns),
            "rows": tabla_faltantes_por_estados.to_dict(orient="records"),
            "excel": "tabla_faltantes_por_estados.xlsx",
        },
    },
    "figures": figures,
}

contenido_js = "window.DASHBOARD_DATA = " + json.dumps(payload, ensure_ascii=False, cls=PlotlyJSONEncoder) + ";\n"
salida_data.write_text(contenido_js, encoding="utf-8")

# Si no existe el HTML origen, se crea desde index.html para que lo puedas editar
if not html_origen.exists() and index_destino.exists():
    html_origen.write_text(index_destino.read_text(encoding="utf-8"), encoding="utf-8")
    print(f"Se creo plantilla HTML: {html_origen}")

if html_origen.exists():
    index_destino.write_text(html_origen.read_text(encoding="utf-8"), encoding="utf-8")
    print(f"Index actualizado desde: {html_origen}")
# else:
#     print(f"No se encontro {html_origen.name}; index.html no se pudo actualizar.")

print(f"Datos exportados en: {salida_data}")
print("Ejecuta esta celda tras modificar graficas o HTML para refrescar el tablero.")

Index actualizado desde: c:\Users\jose.valdez\Downloads\nuevo-inf\informe--de-ang\reporte_interactivo.html
Datos exportados en: c:\Users\jose.valdez\Downloads\nuevo-inf\informe--de-ang\data.js
Ejecuta esta celda tras modificar graficas o HTML para refrescar el tablero.


In [37]:
faltantes_por_estados

,entidad,clues_imb,nombre_de_la_unidad
0,BAJA CALIFORNIA,BCIMB000051,COLONIA LOMA LINDA
1,BAJA CALIFORNIA,BCIMB000063,COLONIA POPULAR 2
2,BAJA CALIFORNIA,BCIMB000075,COLONIA POPULAR 89
3,BAJA CALIFORNIA,BCIMB000104,COL. JALISCO
4,BAJA CALIFORNIA,BCIMB000116,COL. LOMITAS INDECO
...,...,...,...
1153,VERACRUZ DE IGNACIO DE LA LLAVE,VZIMB007032,CENTRO DE SALUD OVIEDO
1154,VERACRUZ DE IGNACIO DE LA LLAVE,VZIMB007633,LA MAJAHUA
1155,VERACRUZ DE IGNACIO DE LA LLAVE,VZIMB008543,C.S. JOPOY
1156,VERACRUZ DE IGNACIO DE LA LLAVE,VZIMB008700,COL. MORELOS
